# SimpleShot Replication, All 4 Models
Press Run All. Trains Conv-4, ResNet-18, DenseNet-121, ResNet-10, sequentially.
Results saved to Drive. Final comparison table at the bottom.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pickle
import os
import time
import gc

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
print('Imports done')

Imports done


In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/CS5782/SimpleShot/SimpleShot Reproduce_WOODY_CHANG'
os.makedirs(BASE_DIR, exist_ok=True)

# Find pkl files
PKL_CANDIDATES = [
    BASE_DIR,
    '/content/drive/MyDrive/SimpleShot',
]
PKL_DIR = None
for d in PKL_CANDIDATES:
    if os.path.exists(os.path.join(d, 'mini-imagenet-cache-train.pkl')):
        PKL_DIR = d
        break
if PKL_DIR is None:
    raise FileNotFoundError('Cannot find pkl files! Check your Drive.')
print(f'Data dir: {PKL_DIR}')
print(f'Base dir: {BASE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data dir: /content/drive/MyDrive/CS5782/SimpleShot/SimpleShot Reproduce_WOODY_CHANG
Base dir: /content/drive/MyDrive/CS5782/SimpleShot/SimpleShot Reproduce_WOODY_CHANG


In [4]:
with open(os.path.join(PKL_DIR, 'mini-imagenet-cache-train.pkl'), 'rb') as f:
    train_data = pickle.load(f)
with open(os.path.join(PKL_DIR, 'mini-imagenet-cache-val.pkl'), 'rb') as f:
    val_data = pickle.load(f)
with open(os.path.join(PKL_DIR, 'mini-imagenet-cache-test.pkl'), 'rb') as f:
    test_data = pickle.load(f)

print(f'Train: {train_data["image_data"].shape}, {len(train_data["class_dict"])} classes')
print(f'Val: {val_data["image_data"].shape}, {len(val_data["class_dict"])} classes')
print(f'Test: {test_data["image_data"].shape}, {len(test_data["class_dict"])} classes')

Train: (38400, 84, 84, 3), 64 classes
Val: (9600, 84, 84, 3), 16 classes
Test: (12000, 84, 84, 3), 20 classes


In [5]:
class MiniImageNetTrainDataset(Dataset):
    def __init__(self, data, transform=None):
        self.images = data['image_data']
        self.class_dict = data['class_dict']
        sorted_classes = sorted(self.class_dict.keys())
        self.label_map = {cls: i for i, cls in enumerate(sorted_classes)}
        self.targets = np.zeros(len(self.images), dtype=int)
        for cls, indices in self.class_dict.items():
            for idx in indices:
                self.targets[idx] = self.label_map[cls]
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.targets[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(84),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

train_dataset = MiniImageNetTrainDataset(train_data, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
print(f'Train loader: {len(train_loader)} batches/epoch')

Train loader: 150 batches/epoch


In [6]:
# ============================================================
# Model Definitions (matching repo)
# ============================================================

# --- Conv-4 ---
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )

class Conv4SimpleShot(nn.Module):
    def __init__(self, num_classes=64):
        super().__init__()
        self.conv1 = conv_block(3, 64)
        self.conv2 = conv_block(64, 64)
        self.conv3 = conv_block(64, 64)
        self.conv4 = conv_block(64, 64)
        self.feat_dim = 1600
        self.classifier = nn.Linear(1600, num_classes)

    def forward(self, x, return_feature=False):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = x.reshape(x.size(0), -1)
        if return_feature:
            return x
        return self.classifier(x)

# --- ResNet-18 ---
def conv3x3(inp, outp, stride=1):
    return nn.Conv2d(inp, outp, 3, stride=stride, padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18SimpleShot(nn.Module):
    def __init__(self, num_classes=64):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(64, 2)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.feat_dim = 512
        self.classifier = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        layers = [BasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x, return_feature=False):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.reshape(x.size(0), -1)
        if return_feature:
            return x
        return self.classifier(x)

# --- DenseNet-121 ---
class DenseNetSimpleShot(nn.Module):
    def __init__(self, num_classes=64):
        super().__init__()
        base = torchvision.models.densenet121(weights=None)
        base.features.conv0 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        base.features.norm0 = nn.Identity()
        base.features.relu0 = nn.Identity()
        base.features.pool0 = nn.Identity()
        self.features = base.features
        self.feat_dim = 1024
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x, return_feature=False):
        features = self.features(x)
        out = F.relu(features, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1)).reshape(features.size(0), -1)
        if return_feature:
            return out
        return self.classifier(out)

MODEL_CLASSES = {
    'conv4': Conv4SimpleShot,
    'resnet18': ResNet18SimpleShot,
    'densenet121': DenseNetSimpleShot,
}
print('All models defined')

All models defined


In [7]:
# ============================================================
# Training + Evaluation Functions
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for i, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
        if (i+1) % 100 == 0:
            print(f'  step {i+1}/{len(loader)}  loss={loss.item():.4f}')
    return total_loss / total, correct / total

def save_checkpoint(model, optimizer, scheduler, epoch, save_dir):
    path = os.path.join(save_dir, f'epoch_{epoch}.pth')
    torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(), 'epoch': epoch}, path)

def load_checkpoint(path, model, optimizer, scheduler, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    # Handle both key formats (v1 vs AUTO)
    model_key = 'model_state_dict' if 'model_state_dict' in ckpt else 'model'
    opt_key = 'optimizer_state_dict' if 'optimizer_state_dict' in ckpt else 'optimizer'
    sched_key = 'scheduler_state_dict' if 'scheduler_state_dict' in ckpt else 'scheduler'
    model.load_state_dict(ckpt[model_key])
    optimizer.load_state_dict(ckpt[opt_key])
    scheduler.load_state_dict(ckpt[sched_key])
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')

def sample_episode(data, n_way=5, k_shot=1, q_query=15):
    class_dict = data['class_dict']
    images = data['image_data']
    selected = random.sample(list(class_dict.keys()), n_way)
    sx, sy, qx, qy = [], [], [], []
    for label, cls in enumerate(selected):
        idxs = random.sample(class_dict[cls], k_shot + q_query)
        sx.append(images[idxs[:k_shot]])
        sy.extend([label] * k_shot)
        qx.append(images[idxs[k_shot:]])
        qy.extend([label] * q_query)
    return np.concatenate(sx), np.array(sy), np.concatenate(qx), np.array(qy)

def extract_features(model, images_np, device, batch_size=256):
    model.eval()
    enlarge_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(int(84 * 256. / 224.)),
        transforms.CenterCrop(84),
        transforms.ToTensor(),
        normalize,
    ])
    feats = []
    with torch.no_grad():
        for i in range(0, len(images_np), batch_size):
            batch = images_np[i:i+batch_size]
            imgs = torch.stack([enlarge_transform(img) for img in batch]).to(device)
            feat = model(imgs, return_feature=True)
            feats.append(feat.cpu().numpy())
    return np.concatenate(feats, axis=0)

def compute_train_mean(model, train_data, device):
    feats = extract_features(model, train_data['image_data'], device)
    return feats.mean(axis=0, keepdims=True)

def l2_normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def apply_transform(features, t, center=None):
    if t == 'UN': return features
    elif t == 'L2N': return l2_normalize(features)
    elif t == 'CL2N': return l2_normalize(features - center)

def evaluate_few_shot(model, data, train_mean, device, k_shot=1, n_episodes=10000, transform_type='CL2N'):
    accs = []
    for i in range(n_episodes):
        sx, sy, qx, qy = sample_episode(data, k_shot=k_shot)
        sf = apply_transform(extract_features(model, sx, device), transform_type, train_mean)
        qf = apply_transform(extract_features(model, qx, device), transform_type, train_mean)
        classes = np.unique(sy)
        centroids = np.stack([sf[sy == c].mean(0) for c in classes])
        dists = ((qf[:, None, :] - centroids[None, :, :]) ** 2).sum(2)
        preds = np.argmin(dists, axis=1)
        accs.append((preds == qy).mean())
        if (i+1) % 1000 == 0:
            print(f'    {i+1}/{n_episodes} episodes done...')
    accs = np.array(accs)
    return accs.mean(), 1.96 * accs.std() / np.sqrt(n_episodes)

print('All functions defined')

Device: cuda
All functions defined


In [ ]:
# ============================================================
# MAIN LOOP: Train + Evaluate all 3 models
# ============================================================

ALL_RESULTS = {}

for model_name in ['conv4', 'resnet18', 'densenet121']:
    print(f'\n{"="*60}')
    print(f'MODEL: {model_name}')
    print(f'{"="*60}')

    ckpt_dir = os.path.join(BASE_DIR, f'checkpoints_{model_name}')
    os.makedirs(ckpt_dir, exist_ok=True)

    # Check if already trained
    best_path = os.path.join(ckpt_dir, 'epoch_best.pth')
    # Special case: DenseNet may have existing checkpoint in checkpoints_256
    if model_name == 'densenet121':
        alt_paths = [
            os.path.join(BASE_DIR, 'checkpoints_256', 'epoch_best.pth'),
            os.path.join('/content/drive/MyDrive/SimpleShot/SimpleShot Reproduce_WOODY_CHANG', 'checkpoints_256', 'epoch_best.pth'),
        ]
        alt_best = None
        for p in alt_paths:
            if os.path.exists(p):
                alt_best = p
                break
        if not os.path.exists(best_path) and alt_best is not None:
            import shutil
            os.makedirs(ckpt_dir, exist_ok=True)
            shutil.copy2(alt_best, best_path)
            print(f'  Copied existing DenseNet checkpoint from checkpoints_256')

    if os.path.exists(best_path):
        print(f'  Checkpoint exists at {best_path}, skipping training.')
    else:
        print(f'  Training {model_name} for 90 epochs...')
        model = MODEL_CLASSES[model_name](num_classes=64).to(device)
        for m in model.modules():
            if isinstance(m, nn.Conv2d):
                if model_name == 'resnet18':
                    nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                else:
                    nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[45, 66], gamma=0.1)
        criterion = nn.CrossEntropyLoss()

        n_params = sum(p.numel() for p in model.parameters())
        print(f'  Parameters: {n_params:,}, Feature dim: {model.feat_dim}')

        best_val_acc = 0.0
        best_epoch = 0
        t0 = time.time()

        for epoch in range(90):
            loss, acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
            print(f'  Epoch {epoch+1}: loss={loss:.4f}, acc={acc:.4f}, lr={optimizer.param_groups[0]["lr"]:.5f}')
            scheduler.step()

            if (epoch + 1) % 4 == 0:
                train_mean = compute_train_mean(model, train_data, device)
                val_acc, _ = evaluate_few_shot(model, val_data, train_mean, device,
                                               k_shot=1, n_episodes=500, transform_type='L2N')
                print(f'    Val 1-shot L2N: {val_acc*100:.2f}%')
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_epoch = epoch + 1
                    save_checkpoint(model, optimizer, scheduler, 'best', ckpt_dir)
                print(f'    Best so far: epoch {best_epoch}, acc {best_val_acc*100:.2f}%')

            save_checkpoint(model, optimizer, scheduler, epoch+1, ckpt_dir)

        elapsed = time.time() - t0
        print(f'  Training complete. Best epoch: {best_epoch}. Time: {elapsed/60:.1f} min')
        del model, optimizer, scheduler, criterion
        torch.cuda.empty_cache()

    # --- Evaluate ---
    print(f'  Evaluating {model_name}...')
    model = MODEL_CLASSES[model_name](num_classes=64).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[45, 66], gamma=0.1)
    load_checkpoint(best_path, model, optimizer, scheduler, device)

    train_mean = compute_train_mean(model, train_data, device)
    results = {}

    for t in ['UN', 'L2N', 'CL2N']:
        for k in [1, 5]:
            print(f'  Evaluating {t} {k}-shot (10,000 episodes)...')
            mean_acc, ci = evaluate_few_shot(model, test_data, train_mean, device,
                                             k_shot=k, n_episodes=10000, transform_type=t)
            results[f'{t}_{k}'] = (mean_acc, ci)
            print(f'    {t} {k}-shot: {mean_acc*100:.2f} +/- {ci*100:.2f}%')

    ALL_RESULTS[model_name] = results

    print(f'\n  {model_name} Results:')
    for t in ['UN', 'L2N', 'CL2N']:
        m1, c1 = results[f'{t}_1']
        m5, c5 = results[f'{t}_5']
        print(f'    {t}  1-shot: {m1*100:.2f}+/-{c1*100:.2f}  5-shot: {m5*100:.2f}+/-{c5*100:.2f}')

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print('\n\nAll models done!')


MODEL: conv4
  Training conv4 for 90 epochs...
  Parameters: 215,552, Feature dim: 1600
  step 100/150  loss=3.5054
  Epoch 1: loss=3.7238, acc=0.1231, lr=0.10000
  step 100/150  loss=3.1353
  Epoch 2: loss=3.2112, acc=0.1972, lr=0.10000
  step 100/150  loss=2.8248
  Epoch 3: loss=2.9982, acc=0.2444, lr=0.10000
  step 100/150  loss=2.7232
  Epoch 4: loss=2.8460, acc=0.2790, lr=0.10000
    Val 1-shot L2N: 37.68%
    Best so far: epoch 4, acc 37.68%
  step 100/150  loss=2.7849
  Epoch 5: loss=2.7401, acc=0.3039, lr=0.10000
  step 100/150  loss=2.5935
  Epoch 6: loss=2.6658, acc=0.3189, lr=0.10000
  step 100/150  loss=2.3508
  Epoch 7: loss=2.5933, acc=0.3389, lr=0.10000
  step 100/150  loss=2.3379
  Epoch 8: loss=2.5361, acc=0.3473, lr=0.10000
    Val 1-shot L2N: 39.22%
    Best so far: epoch 8, acc 39.22%
  step 100/150  loss=2.6369
  Epoch 9: loss=2.5069, acc=0.3553, lr=0.10000
  step 100/150  loss=2.3685
  Epoch 10: loss=2.4461, acc=0.3693, lr=0.10000
  step 100/150  loss=2.4048
  Ep

In [ ]:
# ============================================================
# Final Comparison: All 3 Models vs Paper Table 1 (miniImageNet)
# ============================================================

# Paper Table 1 exact numbers
paper = {
    'conv4':       {'UN_1': (33.17, 0.17), 'UN_5': (63.25, 0.17), 'L2N_1': (48.08, 0.18), 'L2N_5': (66.49, 0.17), 'CL2N_1': (49.69, 0.19), 'CL2N_5': (66.92, 0.17)},
    'resnet18':    {'UN_1': (56.06, 0.20), 'UN_5': (78.63, 0.15), 'L2N_1': (60.16, 0.20), 'L2N_5': (79.94, 0.14), 'CL2N_1': (62.85, 0.20), 'CL2N_5': (80.02, 0.14)},
    'densenet121': {'UN_1': (57.81, 0.21), 'UN_5': (80.43, 0.15), 'L2N_1': (61.49, 0.20), 'L2N_5': (81.48, 0.14), 'CL2N_1': (64.29, 0.20), 'CL2N_5': (81.50, 0.14)},
}

print('=' * 90)
print('SimpleShot Replication: Our Results vs Paper Table 1 (miniImageNet, 5-way, 10K episodes)')
print('=' * 90)
print(f'{"Model":<14} {"Transform":<6} {"1-shot Ours":>12} {"1-shot Paper":>13} {"Gap":>6}  {"5-shot Ours":>12} {"5-shot Paper":>13} {"Gap":>6}')
print('-' * 90)

for mn in ['conv4', 'resnet18', 'densenet121']:
    for t in ['UN', 'L2N', 'CL2N']:
        o1, oc1 = ALL_RESULTS[mn][f'{t}_1']
        o5, oc5 = ALL_RESULTS[mn][f'{t}_5']
        p1, pc1 = paper[mn][f'{t}_1']
        p5, pc5 = paper[mn][f'{t}_5']
        g1 = o1*100 - p1
        g5 = o5*100 - p5
        print(f'{mn:<14} {t:<6} {o1*100:>6.2f}+/-{oc1*100:.2f} {p1:>7.2f}+/-{pc1:.2f} {g1:>+6.2f}  {o5*100:>6.2f}+/-{oc5*100:.2f} {p5:>7.2f}+/-{pc5:.2f} {g5:>+6.2f}')
    print()

print('Note: Negative gap = below paper. Differences due to PyTorch version + pickle data source.')
print('Paper used PyTorch 1.0 + original jpg images. We use PyTorch 2.x + Kaggle pickle.')
print('See GitHub Issue #5: PyTorch 1.4 already causes 2-3% drop vs PyTorch 1.0.')

SimpleShot Replication: Our Results vs Paper Table 1 (miniImageNet, 5-way, 10K episodes)
Model          Transform  1-shot Ours  1-shot Paper    Gap   5-shot Ours  5-shot Paper    Gap
------------------------------------------------------------------------------------------
conv4          UN      34.11+/-0.17   33.17+/-0.17  +0.94   62.19+/-0.17   63.25+/-0.17  -1.06
conv4          L2N     46.34+/-0.18   48.08+/-0.18  -1.74   64.52+/-0.17   66.49+/-0.17  -1.97
conv4          CL2N    47.98+/-0.19   49.69+/-0.19  -1.71   65.12+/-0.17   66.92+/-0.17  -1.80

resnet18       UN      53.90+/-0.21   56.06+/-0.20  -2.16   76.47+/-0.15   78.63+/-0.15  -2.16
resnet18       L2N     58.05+/-0.20   60.16+/-0.20  -2.11   77.59+/-0.15   79.94+/-0.14  -2.35
resnet18       CL2N    60.34+/-0.20   62.85+/-0.20  -2.51   77.61+/-0.15   80.02+/-0.14  -2.41

densenet121    UN      55.40+/-0.20   57.81+/-0.21  -2.41   78.17+/-0.15   80.43+/-0.15  -2.26
densenet121    L2N     59.64+/-0.20   61.49+/-0.20  -1.85  

In [8]:
# ============================================================
# ResNet-10 (additional model)
# Same as ResNet-18 but 1 block per layer instead of 2
# ============================================================

class ResNet10SimpleShot(nn.Module):
    def __init__(self, num_classes=64):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(64, 1)
        self.layer2 = self._make_layer(128, 1, stride=2)
        self.layer3 = self._make_layer(256, 1, stride=2)
        self.layer4 = self._make_layer(512, 1, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.feat_dim = 512
        self.classifier = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        layers = [BasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x, return_feature=False):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.reshape(x.size(0), -1)
        if return_feature:
            return x
        return self.classifier(x)

In [ ]:


# --- Train ResNet-10 ---
model_name = 'resnet10'
print(f'\n{"="*60}')
print(f'MODEL: {model_name}')
print(f'{"="*60}')

ckpt_dir = os.path.join(BASE_DIR, f'checkpoints_{model_name}')
os.makedirs(ckpt_dir, exist_ok=True)
best_path = os.path.join(ckpt_dir, 'epoch_best.pth')

if os.path.exists(best_path):
    print(f'  Checkpoint exists, skipping training.')
else:
    print(f'  Training {model_name} for 90 epochs...')
    model = ResNet10SimpleShot(num_classes=64).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            nn.init.constant_(m.bias, 0)

    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[45, 66], gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}, Feature dim: {model.feat_dim}')

    best_val_acc = 0.0
    best_epoch = 0
    t0 = time.time()

    for epoch in range(90):
        loss, acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        print(f'  Epoch {epoch+1}: loss={loss:.4f}, acc={acc:.4f}, lr={optimizer.param_groups[0]["lr"]:.5f}')
        scheduler.step()

        if (epoch + 1) % 4 == 0:
            train_mean = compute_train_mean(model, train_data, device)
            val_acc, _ = evaluate_few_shot(model, val_data, train_mean, device,
                                           k_shot=1, n_episodes=500, transform_type='L2N')
            print(f'    Val 1-shot L2N: {val_acc*100:.2f}%')
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch + 1
                save_checkpoint(model, optimizer, scheduler, 'best', ckpt_dir)
            print(f'    Best so far: epoch {best_epoch}, acc {best_val_acc*100:.2f}%')

        save_checkpoint(model, optimizer, scheduler, epoch+1, ckpt_dir)

    elapsed = time.time() - t0
    print(f'  Training complete. Best epoch: {best_epoch}. Time: {elapsed/60:.1f} min')
    del model, optimizer, scheduler, criterion
    torch.cuda.empty_cache()

# --- Evaluate ResNet-10 ---
print(f'  Evaluating {model_name}...')
model = ResNet10SimpleShot(num_classes=64).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[45, 66], gamma=0.1)
load_checkpoint(best_path, model, optimizer, scheduler, device)

train_mean = compute_train_mean(model, train_data, device)
results = {}

for t in ['UN', 'L2N', 'CL2N']:
    for k in [1, 5]:
        print(f'  Evaluating {t} {k}-shot (10,000 episodes)...')
        mean_acc, ci = evaluate_few_shot(model, test_data, train_mean, device,
                                         k_shot=k, n_episodes=10000, transform_type=t)
        results[f'{t}_{k}'] = (mean_acc, ci)
        print(f'    {t} {k}-shot: {mean_acc*100:.2f} +/- {ci*100:.2f}%')

ALL_RESULTS['resnet10'] = results

print(f'\n  resnet10 Results:')
for t in ['UN', 'L2N', 'CL2N']:
    m1, c1 = results[f'{t}_1']
    m5, c5 = results[f'{t}_5']
    print(f'    {t}  1-shot: {m1*100:.2f}+/-{c1*100:.2f}  5-shot: {m5*100:.2f}+/-{c5*100:.2f}')

del model, optimizer, scheduler
torch.cuda.empty_cache()


MODEL: resnet10
  Training resnet10 for 90 epochs...
  Parameters: 4,930,944, Feature dim: 512
  step 100/150  loss=3.4658
  Epoch 1: loss=3.6340, acc=0.1135, lr=0.10000
  step 100/150  loss=3.3259
  Epoch 2: loss=3.2836, acc=0.1787, lr=0.10000
  step 100/150  loss=3.0321
  Epoch 3: loss=3.0450, acc=0.2314, lr=0.10000
  step 100/150  loss=2.6572
  Epoch 4: loss=2.8628, acc=0.2726, lr=0.10000
    Val 1-shot L2N: 38.95%
    Best so far: epoch 4, acc 38.95%
  step 100/150  loss=2.8820
  Epoch 5: loss=2.7040, acc=0.3016, lr=0.10000
  step 100/150  loss=2.6185
  Epoch 6: loss=2.5710, acc=0.3355, lr=0.10000
  step 100/150  loss=2.3902
  Epoch 7: loss=2.4440, acc=0.3643, lr=0.10000
  step 100/150  loss=2.4928
  Epoch 8: loss=2.3346, acc=0.3888, lr=0.10000
    Val 1-shot L2N: 42.54%
    Best so far: epoch 8, acc 42.54%
  step 100/150  loss=2.2433
  Epoch 9: loss=2.2313, acc=0.4113, lr=0.10000
  step 100/150  loss=2.1962
  Epoch 10: loss=2.1387, acc=0.4322, lr=0.10000
  step 100/150  loss=1.94

In [10]:
ALL_RESULTS = {}

MODEL_CLASSES_EVAL = {
    'conv4': Conv4SimpleShot,
    'resnet10': ResNet10SimpleShot,
    'resnet18': ResNet18SimpleShot,
    'densenet121': DenseNetSimpleShot,
}

for model_name in ['conv4', 'resnet10', 'resnet18', 'densenet121']:
    print(f'\n{"="*60}')
    print(f'EVALUATING EPOCH 90: {model_name}')
    print(f'{"="*60}')

    if model_name == 'densenet121':
        ckpt_path = os.path.join(BASE_DIR, 'checkpoints_256', 'epoch_90.pth')
    else:
        ckpt_path = os.path.join(BASE_DIR, f'checkpoints_{model_name}', 'epoch_90.pth')

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f'Cannot find: {ckpt_path}')

    model = MODEL_CLASSES_EVAL[model_name](num_classes=64).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[45, 66],
        gamma=0.1
    )

    load_checkpoint(ckpt_path, model, optimizer, scheduler, device)

    train_mean = compute_train_mean(model, train_data, device)
    results = {}

    for t in ['UN', 'L2N', 'CL2N']:
        for k in [1, 5]:
            print(f'  Evaluating {t} {k}-shot...')
            mean_acc, ci = evaluate_few_shot(
                model,
                test_data,
                train_mean,
                device,
                k_shot=k,
                n_episodes=10000,
                transform_type=t
            )
            results[f'{t}_{k}'] = (mean_acc, ci)
            print(f'    {t} {k}-shot: {mean_acc*100:.2f} +/- {ci*100:.2f}%')

    ALL_RESULTS[model_name] = results

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print("\nDone.")


EVALUATING EPOCH 90: conv4
Loaded checkpoint from epoch 90
  Evaluating UN 1-shot...
    1000/10000 episodes done...
    2000/10000 episodes done...
    3000/10000 episodes done...
    4000/10000 episodes done...
    5000/10000 episodes done...
    6000/10000 episodes done...
    7000/10000 episodes done...
    8000/10000 episodes done...
    9000/10000 episodes done...
    10000/10000 episodes done...
    UN 1-shot: 34.28 +/- 0.17%
  Evaluating UN 5-shot...
    1000/10000 episodes done...
    2000/10000 episodes done...
    3000/10000 episodes done...
    4000/10000 episodes done...
    5000/10000 episodes done...
    6000/10000 episodes done...
    7000/10000 episodes done...
    8000/10000 episodes done...
    9000/10000 episodes done...
    10000/10000 episodes done...
    UN 5-shot: 62.35 +/- 0.17%
  Evaluating L2N 1-shot...
    1000/10000 episodes done...
    2000/10000 episodes done...
    3000/10000 episodes done...
    4000/10000 episodes done...
    5000/10000 episodes done.

In [ ]:
# ============================================================
# Final Comparison: All 4 Models vs Paper Table 1 (miniImageNet)(based on best epoch)
# ============================================================

paper = {
    'conv4':       {'UN_1': (33.17, 0.17), 'UN_5': (63.25, 0.17), 'L2N_1': (48.08, 0.18), 'L2N_5': (66.49, 0.17), 'CL2N_1': (49.69, 0.19), 'CL2N_5': (66.92, 0.17)},
    'resnet10':    {'UN_1': (54.45, 0.21), 'UN_5': (76.98, 0.15), 'L2N_1': (57.85, 0.20), 'L2N_5': (78.73, 0.15), 'CL2N_1': (60.85, 0.20), 'CL2N_5': (78.40, 0.15)},
    'resnet18':    {'UN_1': (56.06, 0.20), 'UN_5': (78.63, 0.15), 'L2N_1': (60.16, 0.20), 'L2N_5': (79.94, 0.14), 'CL2N_1': (62.85, 0.20), 'CL2N_5': (80.02, 0.14)},
    'densenet121': {'UN_1': (57.81, 0.21), 'UN_5': (80.43, 0.15), 'L2N_1': (61.49, 0.20), 'L2N_5': (81.48, 0.14), 'CL2N_1': (64.29, 0.20), 'CL2N_5': (81.50, 0.14)},
}

print('=' * 90)
print('SimpleShot Replication: All 4 Models vs Paper Table 1 (miniImageNet, 5-way, 10K episodes)')
print('=' * 90)
print(f'{"Model":<14} {"Transform":<6} {"1-shot Ours":>12} {"1-shot Paper":>13} {"Gap":>6}  {"5-shot Ours":>12} {"5-shot Paper":>13} {"Gap":>6}')
print('-' * 90)

for mn in ['conv4', 'resnet10', 'resnet18', 'densenet121']:
    for t in ['UN', 'L2N', 'CL2N']:
        o1, oc1 = ALL_RESULTS[mn][f'{t}_1']
        o5, oc5 = ALL_RESULTS[mn][f'{t}_5']
        p1, pc1 = paper[mn][f'{t}_1']
        p5, pc5 = paper[mn][f'{t}_5']
        g1 = o1*100 - p1
        g5 = o5*100 - p5
        print(f'{mn:<14} {t:<6} {o1*100:>6.2f}+/-{oc1*100:.2f} {p1:>7.2f}+/-{pc1:.2f} {g1:>+6.2f}  {o5*100:>6.2f}+/-{oc5*100:.2f} {p5:>7.2f}+/-{pc5:.2f} {g5:>+6.2f}')
    print()

print('Note: Negative gap = below paper. Differences due to PyTorch version + pickle data source.')
print('Paper used PyTorch 1.0 + original jpg images. We use PyTorch 2.x + Kaggle pickle.')
print('See GitHub Issue #5: PyTorch 1.4 already causes 2-3% drop vs PyTorch 1.0.')

SimpleShot Replication: All 4 Models vs Paper Table 1 (miniImageNet, 5-way, 10K episodes)
Model          Transform  1-shot Ours  1-shot Paper    Gap   5-shot Ours  5-shot Paper    Gap
------------------------------------------------------------------------------------------
conv4          UN      34.11+/-0.17   33.17+/-0.17  +0.94   62.19+/-0.17   63.25+/-0.17  -1.06
conv4          L2N     46.34+/-0.18   48.08+/-0.18  -1.74   64.52+/-0.17   66.49+/-0.17  -1.97
conv4          CL2N    47.98+/-0.19   49.69+/-0.19  -1.71   65.12+/-0.17   66.92+/-0.17  -1.80

resnet10       UN      52.01+/-0.21   54.45+/-0.21  -2.44   74.39+/-0.16   76.98+/-0.15  -2.59
resnet10       L2N     55.76+/-0.20   57.85+/-0.20  -2.09   75.77+/-0.15   78.73+/-0.15  -2.96
resnet10       CL2N    58.33+/-0.20   60.85+/-0.20  -2.52   75.75+/-0.16   78.40+/-0.15  -2.65

resnet18       UN      53.90+/-0.21   56.06+/-0.20  -2.16   76.47+/-0.15   78.63+/-0.15  -2.16
resnet18       L2N     58.05+/-0.20   60.16+/-0.20  -2.11 

In [11]:
# ============================================================
# Final Comparison: All 4 Models vs Paper Table 2 (miniImageNet)(based on epoch 90)
# ============================================================

paper = {
    'conv4':       {'UN_1': (33.17, 0.17), 'UN_5': (63.25, 0.17), 'L2N_1': (48.08, 0.18), 'L2N_5': (66.49, 0.17), 'CL2N_1': (49.69, 0.19), 'CL2N_5': (66.92, 0.17)},
    'resnet10':    {'UN_1': (54.45, 0.21), 'UN_5': (76.98, 0.15), 'L2N_1': (57.85, 0.20), 'L2N_5': (78.73, 0.15), 'CL2N_1': (60.85, 0.20), 'CL2N_5': (78.40, 0.15)},
    'resnet18':    {'UN_1': (56.06, 0.20), 'UN_5': (78.63, 0.15), 'L2N_1': (60.16, 0.20), 'L2N_5': (79.94, 0.14), 'CL2N_1': (62.85, 0.20), 'CL2N_5': (80.02, 0.14)},
    'densenet121': {'UN_1': (57.81, 0.21), 'UN_5': (80.43, 0.15), 'L2N_1': (61.49, 0.20), 'L2N_5': (81.48, 0.14), 'CL2N_1': (64.29, 0.20), 'CL2N_5': (81.50, 0.14)},
}

print('=' * 90)
print('SimpleShot Replication: All 4 Models vs Paper Table 1 (miniImageNet, 5-way, 10K episodes)')
print('=' * 90)
print(f'{"Model":<14} {"Transform":<6} {"1-shot Ours":>12} {"1-shot Paper":>13} {"Gap":>6}  {"5-shot Ours":>12} {"5-shot Paper":>13} {"Gap":>6}')
print('-' * 90)

for mn in ['conv4', 'resnet10', 'resnet18', 'densenet121']:
    for t in ['UN', 'L2N', 'CL2N']:
        o1, oc1 = ALL_RESULTS[mn][f'{t}_1']
        o5, oc5 = ALL_RESULTS[mn][f'{t}_5']
        p1, pc1 = paper[mn][f'{t}_1']
        p5, pc5 = paper[mn][f'{t}_5']
        g1 = o1*100 - p1
        g5 = o5*100 - p5
        print(f'{mn:<14} {t:<6} {o1*100:>6.2f}+/-{oc1*100:.2f} {p1:>7.2f}+/-{pc1:.2f} {g1:>+6.2f}  {o5*100:>6.2f}+/-{oc5*100:.2f} {p5:>7.2f}+/-{pc5:.2f} {g5:>+6.2f}')
    print()

print('Note: Negative gap = below paper. Differences due to PyTorch version + pickle data source.')
print('Paper used PyTorch 1.0 + original jpg images. We use PyTorch 2.x + Kaggle pickle.')
print('See GitHub Issue #5: PyTorch 1.4 already causes 2-3% drop vs PyTorch 1.0.')

SimpleShot Replication: All 4 Models vs Paper Table 1 (miniImageNet, 5-way, 10K episodes)
Model          Transform  1-shot Ours  1-shot Paper    Gap   5-shot Ours  5-shot Paper    Gap
------------------------------------------------------------------------------------------
conv4          UN      34.28+/-0.17   33.17+/-0.17  +1.11   62.35+/-0.17   63.25+/-0.17  -0.90
conv4          L2N     46.50+/-0.18   48.08+/-0.18  -1.58   64.48+/-0.17   66.49+/-0.17  -2.01
conv4          CL2N    48.03+/-0.19   49.69+/-0.19  -1.66   65.20+/-0.17   66.92+/-0.17  -1.72

resnet10       UN      51.86+/-0.21   54.45+/-0.21  -2.59   74.46+/-0.16   76.98+/-0.15  -2.52
resnet10       L2N     55.68+/-0.20   57.85+/-0.20  -2.17   75.81+/-0.16   78.73+/-0.15  -2.92
resnet10       CL2N    58.39+/-0.20   60.85+/-0.20  -2.46   75.80+/-0.16   78.40+/-0.15  -2.60

resnet18       UN      53.82+/-0.21   56.06+/-0.20  -2.24   76.56+/-0.15   78.63+/-0.15  -2.07
resnet18       L2N     57.72+/-0.20   60.16+/-0.20  -2.44 